In [0]:


# Create the silver schema if it doesn't exist
spark.sql("""
CREATE SCHEMA IF NOT EXISTS cricketscoreprediction.silver
COMMENT 'Silver layer: Cleaned and historized data'
""")
print("✓ Silver schema created")

In [0]:


# Create silver layer table for names with SCD Type 2 structure
spark.sql("""
CREATE TABLE IF NOT EXISTS cricketscoreprediction.silver.names (
  player_key STRING COMMENT 'Unique key: MD5 hash of identifier + name',
  identifier STRING COMMENT 'Player identifier',
  name STRING COMMENT 'Player name',
  valid_from TIMESTAMP COMMENT 'Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'Record valid to timestamp (NULL for current)',
  is_current BOOLEAN COMMENT 'Flag indicating current record',
  checksum STRING COMMENT 'MD5 hash for change detection'
)
USING DELTA
COMMENT 'Silver layer: Player names with SCD Type 2 history tracking'
""")

print("✓ Silver names table created")

In [0]:
# Prepare source data: remove nulls, deduplicate, add keys and checksum
source_names_df = spark.sql("""
WITH cleaned_source AS (
  SELECT 
    identifier,
    name
  FROM cricketscoreprediction.bronze.names
  -- Remove nulls
  WHERE identifier IS NOT NULL 
    AND name IS NOT NULL
    AND TRIM(identifier) != ''
    AND TRIM(name) != ''
),
deduplicated AS (
  -- Remove exact duplicates using ROW_NUMBER
  SELECT 
    identifier,
    name,
    ROW_NUMBER() OVER (
      PARTITION BY identifier, name 
      ORDER BY identifier
    ) AS row_num
  FROM cleaned_source
)
SELECT
  md5(concat_ws('|', identifier, name)) AS player_key,
  identifier,
  name,
  md5(to_json(struct(identifier, name))) AS checksum
FROM deduplicated
WHERE row_num = 1
""")

source_names_df.createOrReplaceTempView("names_source")

print(f"✓ Source data prepared: {source_names_df.count()} records")

In [0]:
# MERGE with SCD Type 2 logic
spark.sql("""
MERGE INTO cricketscoreprediction.silver.names AS target
USING names_source AS source
ON target.player_key = source.player_key AND target.is_current = true

-- When player_key exists but data changed: expire old record
WHEN MATCHED AND target.checksum != source.checksum THEN
  UPDATE SET 
    target.valid_to = current_timestamp(),
    target.is_current = false

-- When player_key doesn't exist: insert new record
WHEN NOT MATCHED THEN
  INSERT (
    player_key, identifier, name, valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.player_key, 
    source.identifier, 
    source.name,
    current_timestamp(), 
    null, 
    true, 
    source.checksum
  )
""")

print("✓ MERGE completed: Current records updated")

In [0]:
# Insert new versions for changed records
spark.sql("""
WITH latest_expired AS (
  SELECT 
    player_key,
    MAX(valid_to) as max_valid_to
  FROM cricketscoreprediction.silver.names
  WHERE is_current = false AND valid_to IS NOT NULL
  GROUP BY player_key
),
changed_records AS (
  SELECT 
    source.player_key,
    source.identifier,
    source.name,
    source.checksum
  FROM names_source AS source
  INNER JOIN cricketscoreprediction.silver.names AS target
    ON source.player_key = target.player_key
    AND target.is_current = false
  INNER JOIN latest_expired
    ON target.player_key = latest_expired.player_key
    AND target.valid_to = latest_expired.max_valid_to
  WHERE source.checksum != target.checksum
)
INSERT INTO cricketscoreprediction.silver.names
SELECT
  player_key,
  identifier,
  name,
  current_timestamp() AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to,
  true AS is_current,
  checksum
FROM changed_records
""")

print("✓ New versions inserted for changed records")

In [0]:
# Validate and display silver layer statistics
stats_df = spark.sql("""
SELECT 
  COUNT(*) as total_records,
  SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) as current_records,
  SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) as historical_records,
  COUNT(DISTINCT identifier) as unique_players,
  COUNT(DISTINCT player_key) as unique_name_combinations
FROM cricketscoreprediction.silver.names
""")

print("\n=== Silver Layer Statistics ===")
display(stats_df)

# Show sample of current records
print("\n=== Sample Current Records ===")
sample_df = spark.sql("""
SELECT 
  identifier,
  name,
  valid_from,
  valid_to,
  is_current
FROM cricketscoreprediction.silver.names
WHERE is_current = true
ORDER BY identifier
LIMIT 10
""")

display(sample_df)

print("\n✓ Silver layer validation complete!")

In [0]:
# Create silver layer table for people with SCD Type 2 structure
spark.sql("""
CREATE TABLE IF NOT EXISTS cricketscoreprediction.silver.people (
  person_key STRING COMMENT 'Unique key: identifier',
  identifier STRING COMMENT 'Person identifier',
  name STRING COMMENT 'Person name',
  unique_name STRING COMMENT 'Unique name',
  key_bcci STRING COMMENT 'BCCI key',
  key_bcci_2 STRING COMMENT 'BCCI key 2',
  key_bigbash INT COMMENT 'Big Bash key',
  key_cricbuzz INT COMMENT 'Cricbuzz key',
  key_cricheroes INT COMMENT 'CricHeroes key',
  key_crichq INT COMMENT 'CricHQ key',
  key_cricinfo INT COMMENT 'Cricinfo key',
  key_cricinfo_2 INT COMMENT 'Cricinfo key 2',
  key_cricinfo_3 INT COMMENT 'Cricinfo key 3',
  key_cricingif INT COMMENT 'Cricingif key',
  key_cricketarchive INT COMMENT 'Cricket Archive key',
  key_cricketarchive_2 INT COMMENT 'Cricket Archive key 2',
  key_cricketworld INT COMMENT 'Cricket World key',
  key_nvplay STRING COMMENT 'NV Play key',
  key_nvplay_2 STRING COMMENT 'NV Play key 2',
  key_opta INT COMMENT 'Opta key',
  key_opta_2 INT COMMENT 'Opta key 2',
  key_pulse INT COMMENT 'Pulse key',
  key_pulse_2 INT COMMENT 'Pulse key 2',
  valid_from TIMESTAMP COMMENT 'Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'Record valid to timestamp (NULL for current)',
  is_current BOOLEAN COMMENT 'Flag indicating current record',
  checksum STRING COMMENT 'MD5 hash for change detection'
)
USING DELTA
COMMENT 'Silver layer: People data with SCD Type 2 history tracking'
""")

print("✓ Silver people table created")

In [0]:
# Prepare source data: remove nulls, deduplicate, add keys and checksum
source_people_df = spark.sql("""
WITH cleaned_source AS (
  SELECT 
    identifier,
    name,
    unique_name,
    key_bcci,
    key_bcci_2,
    key_bigbash,
    key_cricbuzz,
    key_cricheroes,
    key_crichq,
    key_cricinfo,
    key_cricinfo_2,
    key_cricinfo_3,
    key_cricingif,
    key_cricketarchive,
    key_cricketarchive_2,
    key_cricketworld,
    key_nvplay,
    key_nvplay_2,
    key_opta,
    key_opta_2,
    key_pulse,
    key_pulse_2
  FROM cricketscoreprediction.bronze.people
  -- Remove nulls from critical columns only (identifier and name)
  WHERE identifier IS NOT NULL 
    AND name IS NOT NULL
    AND TRIM(identifier) != ''
    AND TRIM(name) != ''
),
deduplicated AS (
  -- Remove exact duplicates using ROW_NUMBER
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY identifier, name, unique_name, key_bcci, key_bcci_2,
                   key_bigbash, key_cricbuzz, key_cricheroes, key_crichq,
                   key_cricinfo, key_cricinfo_2, key_cricinfo_3, key_cricingif,
                   key_cricketarchive, key_cricketarchive_2, key_cricketworld,
                   key_nvplay, key_nvplay_2, key_opta, key_opta_2,
                   key_pulse, key_pulse_2
      ORDER BY identifier
    ) AS row_num
  FROM cleaned_source
)
SELECT
  identifier AS person_key,
  identifier,
  name,
  unique_name,
  key_bcci,
  key_bcci_2,
  key_bigbash,
  key_cricbuzz,
  key_cricheroes,
  key_crichq,
  key_cricinfo,
  key_cricinfo_2,
  key_cricinfo_3,
  key_cricingif,
  key_cricketarchive,
  key_cricketarchive_2,
  key_cricketworld,
  key_nvplay,
  key_nvplay_2,
  key_opta,
  key_opta_2,
  key_pulse,
  key_pulse_2,
  md5(to_json(struct(*))) AS checksum
FROM deduplicated
WHERE row_num = 1
""")

source_people_df.createOrReplaceTempView("people_source")

print(f"✓ Source data prepared: {source_people_df.count()} records")

In [0]:
# MERGE with SCD Type 2 logic
spark.sql("""
MERGE INTO cricketscoreprediction.silver.people AS target
USING people_source AS source
ON target.person_key = source.person_key AND target.is_current = true

-- When person exists but data changed: expire old record
WHEN MATCHED AND target.checksum != source.checksum THEN
  UPDATE SET 
    target.valid_to = current_timestamp(),
    target.is_current = false

-- When person doesn't exist: insert new record
WHEN NOT MATCHED THEN
  INSERT (
    person_key, identifier, name, unique_name,
    key_bcci, key_bcci_2, key_bigbash, key_cricbuzz, key_cricheroes, key_crichq,
    key_cricinfo, key_cricinfo_2, key_cricinfo_3, key_cricingif,
    key_cricketarchive, key_cricketarchive_2, key_cricketworld,
    key_nvplay, key_nvplay_2, key_opta, key_opta_2, key_pulse, key_pulse_2,
    valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.person_key, source.identifier, source.name, source.unique_name,
    source.key_bcci, source.key_bcci_2, source.key_bigbash, source.key_cricbuzz,
    source.key_cricheroes, source.key_crichq, source.key_cricinfo, source.key_cricinfo_2,
    source.key_cricinfo_3, source.key_cricingif, source.key_cricketarchive,
    source.key_cricketarchive_2, source.key_cricketworld, source.key_nvplay,
    source.key_nvplay_2, source.key_opta, source.key_opta_2, source.key_pulse,
    source.key_pulse_2,
    current_timestamp(), null, true, source.checksum
  )
""")

print("✓ MERGE completed: Current records updated")

In [0]:
# Insert new versions for changed records
spark.sql("""
WITH latest_expired AS (
  SELECT 
    person_key,
    MAX(valid_to) as max_valid_to
  FROM cricketscoreprediction.silver.people
  WHERE is_current = false AND valid_to IS NOT NULL
  GROUP BY person_key
),
changed_records AS (
  SELECT 
    source.*
  FROM people_source AS source
  INNER JOIN cricketscoreprediction.silver.people AS target
    ON source.person_key = target.person_key
    AND target.is_current = false
  INNER JOIN latest_expired
    ON target.person_key = latest_expired.person_key
    AND target.valid_to = latest_expired.max_valid_to
  WHERE source.checksum != target.checksum
)
INSERT INTO cricketscoreprediction.silver.people
SELECT
  person_key, identifier, name, unique_name,
  key_bcci, key_bcci_2, key_bigbash, key_cricbuzz, key_cricheroes, key_crichq,
  key_cricinfo, key_cricinfo_2, key_cricinfo_3, key_cricingif,
  key_cricketarchive, key_cricketarchive_2, key_cricketworld,
  key_nvplay, key_nvplay_2, key_opta, key_opta_2, key_pulse, key_pulse_2,
  current_timestamp() AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to,
  true AS is_current,
  checksum
FROM changed_records
""")

print("✓ New versions inserted for changed records")

In [0]:
# Validate and display silver layer statistics for people
stats_people_df = spark.sql("""
SELECT 
  COUNT(*) as total_records,
  SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) as current_records,
  SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) as historical_records,
  COUNT(DISTINCT person_key) as unique_people,
  SUM(CASE WHEN key_cricinfo IS NOT NULL THEN 1 ELSE 0 END) as has_cricinfo_key,
  SUM(CASE WHEN key_cricbuzz IS NOT NULL THEN 1 ELSE 0 END) as has_cricbuzz_key
FROM cricketscoreprediction.silver.people
""")

print("\n=== Silver People Layer Statistics ===")
display(stats_people_df)

# Show sample of current records
print("\n=== Sample Current People Records ===")
sample_people_df = spark.sql("""
SELECT 
  identifier,
  name,
  unique_name,
  key_cricinfo,
  key_cricbuzz,
  valid_from,
  is_current
FROM cricketscoreprediction.silver.people
WHERE is_current = true
ORDER BY identifier
LIMIT 10
""")

display(sample_people_df)

print("\n✓ Silver people layer validation complete!")

In [0]:
# Create silver layer table for innings with flattened delivery columns
spark.sql("""
CREATE TABLE IF NOT EXISTS cricketscoreprediction.silver.matches_innings (
  innings_key STRING COMMENT 'Unique business key for the innings',
  team STRING COMMENT 'Team name',
  super_over BOOLEAN COMMENT 'Is this a super over',
  target_overs DOUBLE COMMENT 'Target overs',
  target_runs BIGINT COMMENT 'Target runs',
  powerplay_from DOUBLE COMMENT 'Powerplay from over',
  powerplay_to DOUBLE COMMENT 'Powerplay to over',
  powerplay_type STRING COMMENT 'Type of powerplay',
  over_number BIGINT COMMENT 'Over number',
  -- Flattened delivery columns
  actual_delivery STRING COMMENT 'Actual delivery number',
  batter STRING COMMENT 'Batter name',
  bowler STRING COMMENT 'Bowler name',
  non_striker STRING COMMENT 'Non-striker name',
  -- Flattened extras
  extras_byes BIGINT COMMENT 'Byes extras',
  extras_legbyes BIGINT COMMENT 'Leg byes extras',
  extras_noballs BIGINT COMMENT 'No balls extras',
  extras_penalty BIGINT COMMENT 'Penalty extras',
  extras_wides BIGINT COMMENT 'Wides extras',
  -- Flattened runs
  runs_batter BIGINT COMMENT 'Runs scored by batter',
  runs_extras BIGINT COMMENT 'Extras runs',
  runs_non_boundary BOOLEAN COMMENT 'Non-boundary flag',
  runs_total BIGINT COMMENT 'Total runs',
  -- Review details (keeping as struct for simplicity)
  review STRUCT<
    batter: STRING,
    by: STRING,
    decision: STRING,
    type: STRING,
    umpire: STRING,
    umpires_call: BOOLEAN
  > COMMENT 'Review details',
  -- Wickets (keeping as array for simplicity)
  wickets ARRAY<STRUCT<
    fielders: ARRAY<STRUCT<name: STRING, substitute: BOOLEAN>>,
    kind: STRING,
    player_out: STRING
  >> COMMENT 'Wickets details',
  -- Replacements (keeping as struct for simplicity)
  replacements STRUCT<
    match: ARRAY<STRUCT<in: STRING, out: STRING, reason: STRING, team: STRING>>,
    role: ARRAY<STRUCT<in: STRING, out: STRING, reason: STRING, role: STRING>>
  > COMMENT 'Replacement details',
  valid_from TIMESTAMP COMMENT 'Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'Record valid to timestamp (NULL for current)',
  is_current BOOLEAN COMMENT 'Flag indicating current record',
  checksum STRING COMMENT 'MD5 hash for change detection'
)
USING DELTA
COMMENT 'Silver layer: Cleaned innings with flattened delivery columns and SCD Type 2'
""")

print("✓ Silver innings table created")

In [0]:
# MERGE clean data from bronze to silver (remove nulls, deduplicate, flatten delivery)
spark.sql("""
MERGE INTO cricketscoreprediction.silver.matches_innings AS target
USING (
WITH cleaned_data AS (
  SELECT 
    innings_key,
    team,
    super_over,
    target_overs,
    target_runs,
    powerplay_from,
    powerplay_to,
    powerplay_type,
    over_number,
    -- Flatten delivery struct
    delivery.actual_delivery,
    delivery.batter,
    delivery.bowler,
    delivery.non_striker,
    delivery.extras.byes as extras_byes,
    delivery.extras.legbyes as extras_legbyes,
    delivery.extras.noballs as extras_noballs,
    delivery.extras.penalty as extras_penalty,
    delivery.extras.wides as extras_wides,
    delivery.runs.batter as runs_batter,
    delivery.runs.extras as runs_extras,
    delivery.runs.non_boundary as runs_non_boundary,
    delivery.runs.total as runs_total,
    delivery.review,
    delivery.wickets,
    delivery.replacements,
    valid_from,
    valid_to,
    is_current,
    checksum
  FROM cricketscoreprediction.bronze.matches_innings_structured
  -- Remove nulls from critical columns
  WHERE innings_key IS NOT NULL 
    AND team IS NOT NULL
    AND delivery.batter IS NOT NULL
    AND delivery.bowler IS NOT NULL
    AND TRIM(innings_key) != ''
    AND TRIM(team) != ''
),
deduplicated AS (
  -- Remove exact duplicates
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY innings_key, checksum, is_current, valid_from
      ORDER BY valid_from
    ) AS row_num
  FROM cleaned_data
)
SELECT
  innings_key,
  team,
  super_over,
  target_overs,
  target_runs,
  powerplay_from,
  powerplay_to,
  powerplay_type,
  over_number,
  actual_delivery,
  batter,
  bowler,
  non_striker,
  extras_byes,
  extras_legbyes,
  extras_noballs,
  extras_penalty,
  extras_wides,
  runs_batter,
  runs_extras,
  runs_non_boundary,
  runs_total,
  review,
  wickets,
  replacements,
  valid_from,
  valid_to,
  is_current,
  checksum
FROM deduplicated
WHERE row_num = 1
) AS source
ON target.innings_key = source.innings_key 
   AND target.checksum = source.checksum
   AND target.valid_from = source.valid_from

-- Only insert records that don't already exist
WHEN NOT MATCHED THEN
  INSERT (
    innings_key, team, super_over, target_overs, target_runs,
    powerplay_from, powerplay_to, powerplay_type, over_number,
    actual_delivery, batter, bowler, non_striker,
    extras_byes, extras_legbyes, extras_noballs, extras_penalty, extras_wides,
    runs_batter, runs_extras, runs_non_boundary, runs_total,
    review, wickets, replacements,
    valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.innings_key, source.team, source.super_over, source.target_overs, source.target_runs,
    source.powerplay_from, source.powerplay_to, source.powerplay_type, source.over_number,
    source.actual_delivery, source.batter, source.bowler, source.non_striker,
    source.extras_byes, source.extras_legbyes, source.extras_noballs, source.extras_penalty, source.extras_wides,
    source.runs_batter, source.runs_extras, source.runs_non_boundary, source.runs_total,
    source.review, source.wickets, source.replacements,
    source.valid_from, source.valid_to, source.is_current, source.checksum
  )
""")

print("✓ Clean innings data merged to silver layer with flattened delivery columns (safe incremental update)")

In [0]:
# Validate and display silver layer statistics for innings
stats_innings_df = spark.sql("""
SELECT 
  COUNT(*) as total_records,
  SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) as current_records,
  SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) as historical_records,
  COUNT(DISTINCT innings_key) as unique_innings,
  COUNT(DISTINCT team) as unique_teams,
  SUM(CASE WHEN super_over = true THEN 1 ELSE 0 END) as super_over_deliveries,
  SUM(CASE WHEN target_runs IS NOT NULL THEN 1 ELSE 0 END) as with_target,
  SUM(runs_total) as total_runs_scored
FROM cricketscoreprediction.silver.matches_innings
""")

print("\n=== Silver Innings Layer Statistics ===")
display(stats_innings_df)

# Show sample of current records with flattened columns
print("\n=== Sample Current Innings Records (Flattened) ===")
sample_innings_df = spark.sql("""
SELECT 
  innings_key,
  team,
  over_number,
  actual_delivery,
  batter,
  bowler,
  non_striker,
  runs_total,
  runs_batter,
  extras_wides,
  super_over,
  is_current
FROM cricketscoreprediction.silver.matches_innings
WHERE is_current = true
ORDER BY innings_key
LIMIT 10
""")

display(sample_innings_df)

print("\n✓ Silver innings layer validation complete!")

In [0]:
# Drop existing table first
spark.sql("DROP TABLE IF EXISTS cricketscoreprediction.silver.matches_info")
print("✓ Existing table dropped")

# Create silver layer table for matches info with exploded array fields
spark.sql("""
CREATE TABLE cricketscoreprediction.silver.matches_info (
  match_key STRING COMMENT 'Unique business key for the match',
  city STRING COMMENT 'City where match was played',
  match_date STRING COMMENT 'Primary match date',
  match_type STRING COMMENT 'Type of match (T20, ODI, Test)',
  team1 STRING COMMENT 'First team',
  team2 STRING COMMENT 'Second team',
  toss_decision STRING COMMENT 'Toss decision (bat/field)',
  toss_winner STRING COMMENT 'Team that won the toss',
  umpire1 STRING COMMENT 'First umpire',
  umpire2 STRING COMMENT 'Second umpire',
  umpire3 STRING COMMENT 'Third umpire (if any)',
  venue STRING COMMENT 'Venue name',
  player_of_match STRING COMMENT 'Player of the match',
  outcome_result STRING COMMENT 'Match result type',
  outcome_winner STRING COMMENT 'Winning team',
  outcome_by_runs BIGINT COMMENT 'Victory margin by runs',
  outcome_by_wickets BIGINT COMMENT 'Victory margin by wickets',
  overs BIGINT COMMENT 'Total overs',
  valid_from TIMESTAMP COMMENT 'Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'Record valid to timestamp (NULL for current)',
  is_current BOOLEAN COMMENT 'Flag indicating current record',
  checksum STRING COMMENT 'MD5 hash for change detection'
)
USING DELTA
COMMENT 'Silver layer: Cleaned match information with exploded arrays and SCD Type 2 history'
""")

print("✓ Silver matches info table created")

In [0]:
# MERGE clean data from bronze to silver (remove nulls, deduplicate, explode arrays)
spark.sql("""
MERGE INTO cricketscoreprediction.silver.matches_info AS target
USING (
  WITH cleaned_data AS (
    SELECT 
      match_key,
      city,
      -- Explode arrays to individual columns (using try_element_at for safe access)
      try_element_at(dates, 1) as match_date,
      match_type,
      try_element_at(teams, 1) as team1,
      try_element_at(teams, 2) as team2,
      toss_decision,
      toss_winner,
      try_element_at(umpires, 1) as umpire1,
      try_element_at(umpires, 2) as umpire2,
      try_element_at(umpires, 3) as umpire3,
      venue,
      try_element_at(player_of_match, 1) as player_of_match,
      outcome_result,
      outcome_winner,
      outcome_by_runs,
      outcome_by_wickets,
      overs,
      valid_from,
      valid_to,
      is_current,
      checksum
    FROM cricketscoreprediction.bronze.matches_info_structured
    -- Remove nulls from critical columns
    WHERE match_key IS NOT NULL 
      AND TRIM(match_key) != ''
  ),
  deduplicated AS (
    -- Remove exact duplicates
    SELECT 
      *,
      ROW_NUMBER() OVER (
        PARTITION BY match_key, checksum, is_current, valid_from
        ORDER BY valid_from
      ) AS row_num
    FROM cleaned_data
  )
  SELECT
    match_key,
    city,
    match_date,
    match_type,
    team1,
    team2,
    toss_decision,
    toss_winner,
    umpire1,
    umpire2,
    umpire3,
    venue,
    player_of_match,
    outcome_result,
    outcome_winner,
    outcome_by_runs,
    outcome_by_wickets,
    overs,
    valid_from,
    valid_to,
    is_current,
    checksum
  FROM deduplicated
  WHERE row_num = 1
) AS source
ON target.match_key = source.match_key 
   AND target.checksum = source.checksum
   AND target.valid_from = source.valid_from

-- Only insert records that don't already exist
WHEN NOT MATCHED THEN
  INSERT (
    match_key, city, match_date, match_type, team1, team2, toss_decision, toss_winner,
    umpire1, umpire2, umpire3, venue, player_of_match, outcome_result, outcome_winner,
    outcome_by_runs, outcome_by_wickets, overs,
    valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.match_key, source.city, source.match_date, source.match_type, source.team1, source.team2,
    source.toss_decision, source.toss_winner, source.umpire1, source.umpire2, source.umpire3, source.venue,
    source.player_of_match, source.outcome_result, source.outcome_winner,
    source.outcome_by_runs, source.outcome_by_wickets, source.overs,
    source.valid_from, source.valid_to, source.is_current, source.checksum
  )
""")

print("✓ Clean matches info data merged to silver layer (safe incremental update)")

In [0]:
# Validate and display silver layer statistics for matches info (exploded arrays)
stats_matches_df = spark.sql("""
SELECT 
  COUNT(*) as total_records,
  SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) as current_records,
  SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) as historical_records,
  COUNT(DISTINCT match_key) as unique_matches,
  COUNT(DISTINCT match_type) as match_types,
  COUNT(DISTINCT venue) as unique_venues,
  COUNT(DISTINCT team1) + COUNT(DISTINCT team2) as total_unique_teams,
  SUM(CASE WHEN outcome_winner IS NOT NULL THEN 1 ELSE 0 END) as matches_with_winner,
  SUM(CASE WHEN umpire3 IS NOT NULL THEN 1 ELSE 0 END) as matches_with_third_umpire
FROM cricketscoreprediction.silver.matches_info
""")

print("\n=== Silver Matches Info Layer Statistics ===")
display(stats_matches_df)

# Show sample of current records with exploded arrays
print("\n=== Sample Current Match Records (Exploded Arrays) ===")
sample_matches_df = spark.sql("""
SELECT 
  match_key,
  city,
  match_date,
  match_type,
  team1,
  team2,
  venue,
  umpire1,
  umpire2,
  player_of_match,
  toss_winner,
  outcome_winner,
  outcome_by_runs,
  outcome_by_wickets,
  is_current
FROM cricketscoreprediction.silver.matches_info
WHERE is_current = true
ORDER BY match_key
LIMIT 10
""")

display(sample_matches_df)

print("\n✓ Silver matches info layer validation complete!")